# Multimode phase retrieval

Minimal incoherent multimode reconstruction using `library/phase_retrieval_core_multimode.py`. The measured Fourier intensity is modeled as the sum of the modal intensities.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from library import phase_retrieval_core_multimode as phr_multi

## Load prepared arrays

The same 2D measured intensities and masks used by the standard reconstruction can be used here. A 2D support is shared by all modes; alternatively provide `(Nmodes, nx, ny)` supports.

In [ ]:
data = np.load(Path("data/phase_retrieval_inputs.npz"))
pos = data["pos"]
neg = data["neg"]
mask_pixel = data["mask_pixel"]
supportmask = data["supportmask"]

Nmodes = 3

In [ ]:
recipe = {
    "Nmodes": Nmodes,
    "algorithm_list": ["HAPRE", "ER", "ER"],
    "number_iterations": [500, 100, 100],
    "helicity": ["pos", "pos", "neg"],
    "beta_zero": [0.5, 0.5, 0.5],
    "beta_mode": ["arctan", "const", "const"],
    "alpha_zero": [0.0, 0.0, 0.0],
    "alpha_mode": ["const", "const", "const"],
    "RL_its": [0, 0, 0],
    "RL_freqs": [1e9, 1e9, 1e9],
    "TV_freqs": [1e9, 1e9, 1e9],
    "plot_every": [100, 50, 50],
    "average_img": [20, 20, 20],
    "Fourier_last": [True, True, True],
    "Startimage": [None, "pos", "pos"],
    "Startgamma": [None, None, None],
    "hologram_intensity_cutoff_vmin": -1,
}

In [ ]:
(
    retrieved_pos,
    retrieved_neg,
    retrieved_pos_pc,
    retrieved_neg_pc,
    bsmask_pos,
    bsmask_neg,
    gamma_pos,
    gamma_neg,
    error,
) = phr_multi.phase_retrieval_algorithm(
    pos,
    neg,
    mask_pixel,
    supportmask,
    phase_retrieval_recipe=recipe,
)

phr_multi.plot_phase_retrieval_errors(error, recipe)
plt.show()

## Inspect the reconstructed modes

For `Nmodes > 1`, each returned reconstruction has shape `(Nmodes, nx, ny)`.

In [ ]:
object_modes_pos = np.stack([
    np.fft.fft2(np.fft.fftshift(mode)) for mode in retrieved_pos
])
object_modes_neg = np.stack([
    np.fft.fft2(np.fft.fftshift(mode)) for mode in retrieved_neg
])

fig, axes = plt.subplots(2, Nmodes, figsize=(4 * Nmodes, 7))
for mode in range(Nmodes):
    axes[0, mode].imshow(np.abs(object_modes_pos[mode]), cmap="gray")
    axes[0, mode].set_title(f"Positive mode {mode}")
    axes[1, mode].imshow(np.abs(object_modes_neg[mode]), cmap="gray")
    axes[1, mode].set_title(f"Negative mode {mode}")
for ax in axes.ravel():
    ax.axis("off")
plt.show()

modal_fourier_intensity_pos = np.sum(np.abs(retrieved_pos) ** 2, axis=0)